In [ ]:
# Diffusion model dependencies (TabDDPM + CoDi)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# CoDi: ChaejeongLee/CoDi (_vendor/CoDi)
%pip install -q ForestDiffusion xgboost category-encoders libzero rtdl imbalanced-learn absl-py tensorboardX

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "goggle" / "src"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "CoDi"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_codi


In [ ]:
pip install ucimlrepo

In [ ]:
pip install sdv

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



# Load dataset
breast_cancer = fetch_ucirepo(id=17)

X = breast_cancer.data.features
y = breast_cancer.data.targets

cancer_data = pd.concat([X, y], axis=1)

target_col = cancer_data.columns[-1]

# IMPORTANT: Do NOT fit generators on the full dataset
# Split first to avoid data leakage

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(cancer_data)

scores = {
    "TabDDPM": [],
    "CoDi": [],
    "CTGAN": [],
    "CopulaGAN": [],
    "TVAE": [],
    "GaussianCopula": []
}

# Initialize dictionary to store all generated synthetic dataframes across all runs
synthetic_datasets = {
    "TabDDPM": [],
    "CoDi": [],
    "CTGAN": [],
    "CopulaGAN": [],
    "TVAE": [],
    "GaussianCopula": []
}


N_RUNS = 10
N_SAMPLES = 1000
TEST_SIZE = 0.2

In [ ]:
for seed in range(N_RUNS):

    print(f"\n================ RUN {seed+1}/{N_RUNS} ================")

    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # ---------------------------------------------------
    # TRAIN / TEST SPLIT (NO LEAKAGE)
    # ---------------------------------------------------

    train_real, test_real = train_test_split(
        cancer_data,
        test_size=0.2,
        stratify=cancer_data[target_col],
        random_state=seed
    )

    train_metadata = SingleTableMetadata()
    train_metadata.detect_from_dataframe(train_real)

    # ---------------------------------------------------

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)


In [ ]:
# CoDi

try:
    import traceback

    print("Training CoDi...")
    synthetic_codi = train_codi(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["CoDi"] = synthetic_codi.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_codi,
        metadata=train_metadata,
    )

    scores["CoDi"] = quality.get_score()

    print("CoDi:", round(scores["CoDi"], 4))

    del synthetic_codi

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("CoDi Failed:")
    traceback.print_exc()


In [ ]:
# ---------------------------------------------------
# SDV MODELS
# ---------------------------------------------------

models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(
            N_SAMPLES
        )

        # Store synthetic data for this run
        synthetic_datasets[model_name].append(synthetic_data.copy())

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        score = quality.get_score()

        scores[model_name].append(score)

        print(
            f"Run {seed+1}/{N_RUNS} | "
            f"{model_name}: {round(score,4)}"
        )

    except Exception as e:

        print(
            f"Run {seed+1}/{N_RUNS} | "
            f"{model_name} Failed: {e}"
        )

In [ ]:
# ==========================================================
# STORE ALL SYNTHETIC DATASETS GENERATED ACROSS 10 RUNS
# (Confirmation of collected data)
# ==========================================================

# The synthetic_datasets dictionary should now be populated from the generation loops.

print("--- Summary of Stored Synthetic Data Across Runs ---")
for model_name, datasets in synthetic_datasets.items():
    print(
        f"{model_name}: Stored {len(datasets)} datasets (expected {N_RUNS} datasets)."
    )
    if datasets:
        print(f"  Example head from first DataFrame for {model_name}:\n{datasets[0].head()}\n")
    else:
        print(f"  No datasets stored for {model_name}.\n")

# Now, synthetic_datasets is ready for TSTR evaluation in a subsequent cell if needed.
# X_real and y_real should be defined before any TSTR evaluation that uses them.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Re-define models to ensure it contains scikit-learn classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [ ]:
# ----------------------------------------------------
# TRTR (Train Real, Test Real) Evaluation
# ----------------------------------------------------
print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")
trtr_results = []

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"  Running {model_name} for TRTR...")

    for seed in range(N_RUNS):

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(accuracy_score(y_test_real, y_pred))
        f1_scores.append(f1_score(y_test_real, y_pred, average="weighted", zero_division=0))
        precision_scores.append(precision_score(y_test_real, y_pred, average="weighted", zero_division=0))
        recall_scores.append(recall_score(y_test_real, y_pred, average="weighted", zero_division=0))

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": np.mean(accuracy_scores),
        "Accuracy Std_TRTR": np.std(accuracy_scores),
        "F1 Mean_TRTR": np.mean(f1_scores),
        "F1 Std_TRTR": np.std(f1_scores),
        "Precision Mean_TRTR": np.mean(precision_scores),
        "Precision Std_TRTR": np.std(precision_scores),
        "Recall Mean_TRTR": np.mean(recall_scores),
        "Recall Std_TRTR": np.std(recall_scores),
        "Accuracy (Mean±Std)_TRTR": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
        "F1 (Mean±Std)_TRTR": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
        "Precision (Mean±Std)_TRTR": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
        "Recall (Mean±Std)_TRTR": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)
display(trtr_results_df[
    [
        "Model",
        "Accuracy (Mean±Std)_TRTR",
        "F1 (Mean±Std)_TRTR",
        "Precision (Mean±Std)_TRTR",
        "Recall (Mean±Std)_TRTR"
    ]
])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

def evaluate_models(train_df, test_df, label_col, models, test_size=0.2, seed=42):
    # Split TRAIN df -> (train part only)  (we don’t need train_df's test split)
    X_train = train_df.drop(columns=[label_col])
    y_train = train_df[label_col]

    X_train, _, y_train, _ = train_test_split(
        X_train, y_train, test_size=test_size, random_state=seed, stratify=y_train
    )

    # Split TEST df -> (test part only)
    X_test = test_df.drop(columns=[label_col])
    y_test = test_df[label_col]

    _, X_test, _, y_test = train_test_split(
        X_test, y_test, test_size=test_size, random_state=seed, stratify=y_test
    )

    # Scale using TRAIN statistics only
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s  = scaler.transform(X_test)

    rows = []
    for name, clf in models.items():
        clf.fit(X_train_s, y_train)

        y_pred = clf.predict(X_test_s)

        # AUC needs probabilities (or decision_function); handle both safely
        if hasattr(clf, "predict_proba"):
            y_prob = clf.predict_proba(X_test_s)[:, 1]
        elif hasattr(clf, "decision_function"):
            scores = clf.decision_function(X_test_s)
            # convert to 0-1-ish range (not perfect prob, but works for AUC)
            y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
        else:
            y_prob = None

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, pos_label='M', average='binary', zero_division=0)
        precision = precision_score(y_test, y_pred, pos_label='M', average='binary', zero_division=0)
        recall = recall_score(y_test, y_pred, pos_label='M', average='binary', zero_division=0)

        auc = roc_auc_score(y_test, y_prob) if y_prob is not None else float("nan")

        rows.append({"Model": name, "Accuracy": acc, "F1": f1, "Precision": precision, "Recall": recall, "AUC": auc})

    return pd.DataFrame(rows).sort_values(by="AUC", ascending=False)

In [ ]:
import pandas as pd
import numpy as np # Need numpy for mean/std

label_col = "Diagnosis"
model_order = ["CTGAN", "CopulaGAN", "TVAE", "GaussianCopula", "CoDi", "TabDDPM"]

# The existing trtr_results calculation (single evaluation)
trtr_results = evaluate_models(
    train_df=cancer_data,
    test_df= cancer_data,
    label="diagnosis",
    models=models
)

print("TRTR (Train Real, Test Real)")
display(trtr_results)
print("=" * 70)

all_comparisons = []

for synth_name in model_order:
    print(f"{synth_name} - TSTR (Train Synthetic, Test Real) Evaluation Across {N_RUNS} Runs")

    # Dictionary to collect results for each classifier across all N_RUNS for the current synth_name
    # Each key (classifier name) will hold a list of scores for each metric (from each run)
    classifier_run_metrics = {
        clf_name: {"Accuracy": [], "F1": [], "Precision": [], "Recall": [], "AUC": []}
        for clf_name in models.keys()
    }

    # Iterate through each synthetic dataset generated for this synthetic model across N_RUNS
    # `synthetic_datasets[synth_name]` is a list of N_RUNS dataframes
    for run_idx, synthetic_train_df_for_run in enumerate(synthetic_datasets[synth_name]):
        # Evaluate models using the synthetic data from one specific run as training data
        # and the real data (split internally by evaluate_models) as test data.
        run_results = evaluate_models(
            train_df=synthetic_train_df_for_run, # This is now a single DataFrame
            test_df=cancer_data,                 # Full real data for internal splitting
            label="diagnosis",
            models=models,
            test_size=TEST_SIZE,
            seed=run_idx                         # Use run_idx for seed for reproducibility of internal splits
        )

        # Collect metrics for each classifier from this run
        for _, row in run_results.iterrows():
            clf_name = row["Model"]
            classifier_run_metrics[clf_name]["Accuracy"].append(row["Accuracy"])
            classifier_run_metrics[clf_name]["F1"].append(row["F1"])
            classifier_run_metrics[clf_name]["Precision"].append(row["Precision"])
            classifier_run_metrics[clf_name]["Recall"].append(row["Recall"])
            classifier_run_metrics[clf_name]["AUC"].append(row["AUC"])

    # Calculate the mean of the metrics across all N_RUNS for each classifier for the current synthetic model
    tstr_mean_metrics = []
    for clf_name, metrics_list in classifier_run_metrics.items():
        tstr_mean_metrics.append({
            "Model": clf_name,
            "Accuracy": np.mean(metrics_list["Accuracy"]),
            "F1": np.mean(metrics_list["F1"]),
            "Precision": np.mean(metrics_list["Precision"]),
            "Recall": np.mean(metrics_list["Recall"]),
            "AUC": np.mean(metrics_list["AUC"])
        })

    # Convert to DataFrame for display and comparison
    tstr_results = pd.DataFrame(tstr_mean_metrics).sort_values(by="AUC", ascending=False)


    print(f"{synth_name} - TSTR (Train Synthetic, Test Real)")
    display(tstr_results)

    comparison = trtr_results.merge(
        tstr_results, on="Model", suffixes=('_TRTR', '_TSTR')
    )

    comparison["AUC_Drop"] = comparison["AUC_TRTR"] - comparison["AUC_TSTR"]

    if "F1_TRTR" in comparison.columns and "F1_TSTR" in comparison.columns:
        comparison["F1_Drop"] = comparison["F1_TRTR"] - comparison["F1_TSTR"]
    if "Accuracy_TRTR" in comparison.columns and "Accuracy_TSTR" in comparison.columns:
        comparison["Accuracy_Drop"] = comparison["Accuracy_TRTR"] - comparison["Accuracy_TSTR"]

    comparison["Synthetic_Model"] = synth_name
    comparison = comparison.sort_values("AUC_Drop", ascending=False)

    print(f"{synth_name} - TRTR vs TSTR Comparison")
    display(comparison)
    print("=" * 70)

    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)

summary = (combined_comparison
           .groupby("Synthetic_Model", as_index=False)["AUC_Drop"]
           .mean()
           .sort_values("AUC_Drop"))

print("Average AUC drop by synthetic generator (lower is better):")
display(summary)


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Re-define models to ensure it contains scikit-learn classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

# Assume cancer_data and target_col are already defined globally
X_real = cancer_data.drop(columns=[target_col])
y_real = cancer_data[target_col]

N_RUNS = 10 # Number of runs for evaluation, consistent with prior cells
TEST_SIZE = 0.3 # Test set size for train_test_split

# ----------------------------------------------------
# TRTR (Train Real, Test Real) Evaluation
# ----------------------------------------------------
print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")
trtr_results = []

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"  Running {model_name} for TRTR...")

    for seed in range(N_RUNS):

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X_real,
            y_real,
            test_size=TEST_SIZE,
            stratify=y_real,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(accuracy_score(y_test_real, y_pred))
        f1_scores.append(f1_score(y_test_real, y_pred, average="weighted", zero_division=0))
        precision_scores.append(precision_score(y_test_real, y_pred, average="weighted", zero_division=0))
        recall_scores.append(recall_score(y_test_real, y_pred, average="weighted", zero_division=0))

    trtr_results.append({
        "Model": model_name,
        "Accuracy Mean_TRTR": np.mean(accuracy_scores),
        "Accuracy Std_TRTR": np.std(accuracy_scores),
        "F1 Mean_TRTR": np.mean(f1_scores),
        "F1 Std_TRTR": np.std(f1_scores),
        "Precision Mean_TRTR": np.mean(precision_scores),
        "Precision Std_TRTR": np.std(precision_scores),
        "Recall Mean_TRTR": np.mean(recall_scores),
        "Recall Std_TRTR": np.std(recall_scores),
        "Accuracy (Mean±Std)_TRTR": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
        "F1 (Mean±Std)_TRTR": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
        "Precision (Mean±Std)_TRTR": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
        "Recall (Mean±Std)_TRTR": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results)
display(trtr_results_df[
    [
        "Model",
        "Accuracy (Mean±Std)_TRTR",
        "F1 (Mean±Std)_TRTR",
        "Precision (Mean±Std)_TRTR",
        "Recall (Mean±Std)_TRTR"
    ]
])


In [ ]:
# ----------------------------------------------------
# TSTR (Train Synthetic, Test Real) Evaluation
# (Without AUC Metric)
# ----------------------------------------------------
print("\n--- Starting TSTR Evaluation (Train Synthetic, Test Real) ---")

all_tstr_results = []

if 'synthetic_datasets' not in locals() and 'synthetic_datasets' not in globals():
    print("Warning: 'synthetic_datasets' variable not found. TSTR evaluation will be skipped.")

elif not synthetic_datasets:
    print("Warning: 'synthetic_datasets' is empty. TSTR evaluation will be skipped.")

else:
    for synth_data_name, list_of_synthetic_dfs in synthetic_datasets.items():

        print(f"\nEvaluating TSTR for: {synth_data_name}")

        classifier_run_metrics = {
            clf_name: {
                "Accuracy": [],
                "F1": [],
                "Precision": [],
                "Recall": []
            }
            for clf_name in models.keys()
        }

        for run_idx, synthetic_train_df in enumerate(list_of_synthetic_dfs):

            run_results_df = evaluate_models(
                train_df=synthetic_train_df,
                test_df=cancer_data,
                label_col=target_col,
                models=models,
                test_size=TEST_SIZE,
                seed=run_idx
            )

            for _, row in run_results_df.iterrows():
                clf_name = row["Model"]

                classifier_run_metrics[clf_name]["Accuracy"].append(row["Accuracy"])
                classifier_run_metrics[clf_name]["F1"].append(row["F1"])
                classifier_run_metrics[clf_name]["Precision"].append(row["Precision"])
                classifier_run_metrics[clf_name]["Recall"].append(row["Recall"])

        # Aggregate Mean ± Std across runs
        for clf_name, metrics in classifier_run_metrics.items():

            acc_mean = np.mean(metrics["Accuracy"])
            acc_std = np.std(metrics["Accuracy"])

            f1_mean = np.mean(metrics["F1"])
            f1_std = np.std(metrics["F1"])

            prec_mean = np.mean(metrics["Precision"])
            prec_std = np.std(metrics["Precision"])

            rec_mean = np.mean(metrics["Recall"])
            rec_std = np.std(metrics["Recall"])

            all_tstr_results.append({
                "Synthetic_Model": synth_data_name,
                "Model": clf_name,

                "Accuracy Mean_TSTR": acc_mean,
                "Accuracy Std_TSTR": acc_std,

                "F1 Mean_TSTR": f1_mean,
                "F1 Std_TSTR": f1_std,

                "Precision Mean_TSTR": prec_mean,
                "Precision Std_TSTR": prec_std,

                "Recall Mean_TSTR": rec_mean,
                "Recall Std_TSTR": rec_std,

                "Accuracy (Mean±Std)_TSTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
                "F1 (Mean±Std)_TSTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
                "Precision (Mean±Std)_TSTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
                "Recall (Mean±Std)_TSTR": f"{rec_mean:.4f} ± {rec_std:.4f}",
            })

    tstr_results_df = pd.DataFrame(all_tstr_results)

    display(
        tstr_results_df[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy (Mean±Std)_TSTR",
                "F1 (Mean±Std)_TSTR",
                "Precision (Mean±Std)_TSTR",
                "Recall (Mean±Std)_TSTR",
            ]
        ].sort_values(["Synthetic_Model", "Model"])
    )